## Visualizar OACNN TEST

In [1]:
def desplazar_nube(pcd_gt, pcd_pred, points, offset):
    """Desplaza la nube de puntos predicha (pcd_pred) en el eje X y ajusta su altura (Z) para mantener la coherencia con el terreno."""
    min_x = points[:, 0].min()
    max_x = points[:, 0].max()
    ancho_real = max_x - min_x

    # Calcular la diferencia de altura (Delta Z) entre los extremos del eje X
    # Tomamos una pequeña franja (5%) en el inicio y el final para promediar la altura
    mask_inicio = points[:, 0] < (min_x + ancho_real * 0.05)
    mask_final = points[:, 0] > (max_x - ancho_real * 0.05)

    z_promedio_inicio = points[mask_inicio, 2].mean()
    z_promedio_final = points[mask_final, 2].mean()

    # Calcular la pendiente (m = delta_z / delta_x)
    # Esto nos dice cuántos metros sube/baja el terreno por cada metro que avanzamos en X
    pendiente = (z_promedio_final - z_promedio_inicio) / ancho_real

    # Calcular el desplazamiento en Z basado en el offset de X
    # Si movemos la nube 'offset' metros en X, debemos moverla 'offset * pendiente' en Z
    z_offset = pendiente * offset

    # Aplicar la traslación 3D (X y Z sincronizados)
    pcd_pred.translate([offset, 0, z_offset])

### VISUALIZAR GT Y PREDICCIONES LADO A LADO

In [47]:
import laspy
import numpy as np
import open3d as o3d
import torch
import matplotlib.pyplot as plt

def visualizacion_lado_a_lado(path_las, pred_pth):

    try:
        preds = torch.load(pred_pth)
    except Exception as e:
        print(f"Error al cargar el archivo .pth: {e}")
        return
    


    # Cargar el LAS
    las = laspy.read(path_las)
    points = np.vstack((las.x, las.y, las.z)).T
    gt_labels = np.array(las.classification)

    centro = points.mean(axis=0)
    points = points - centro


    # Crear nube de puntos de Open3D
    pcd_gt = o3d.geometry.PointCloud()
    pcd_gt.points = o3d.utility.Vector3dVector(points)

    # Colorear Ground Truth (IDs >= 3)
    max_label = int(gt_labels.max())
    colors_dict = np.random.uniform(0, 1, size=(max_label + 1, 3))
    colors_dict[:3] = [0.2, 0.2, 0.2]  # Fondo (0,1,2) en gris
    pcd_gt.colors = o3d.utility.Vector3dVector(colors_dict[gt_labels])

    # Crear la "Predicción" para el lado derecho
    pcd_pred = o3d.geometry.PointCloud(pcd_gt)

    # Crear array de etiquetas
    num_puntos = points.shape[0]
    num_instancias = preds['pred_masks'].shape[0]
    labels = np.full(num_puntos, -1, dtype=np.int32)

    # IDs únicos a cada máscara
    for i in range(num_instancias):
        mask = preds['pred_masks'][i].cpu().numpy().astype(bool)
        labels[mask] = i

    # Colorear la predicción según las etiquetas asignadas
    cmap = plt.get_cmap("tab20")
    colors = np.zeros((num_puntos, 3))
    
    # Color para el fondo/puntos no instanciados (Gris muy oscuro)
    colors[labels == -1] = [0.15, 0.15, 0.15]
    
    # Color para los árboles detectados
    for i in range(num_instancias):
        colors[labels == i] = cmap(i % 20)[:3]
    pcd_pred.colors = o3d.utility.Vector3dVector(colors)
    

    desplazar_nube(pcd_gt, pcd_pred, points, offset=25)  # Desplazamos 20 metros en X

    # Visualizar
    print(f"Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)")
    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name="Comparativa de Instancias - Santomera", width=1200, height=800)
    
    vis.add_geometry(pcd_gt)
    #vis.add_geometry(pcd_pred)
    
    
    opt = vis.get_render_option()
    opt.background_color = np.asarray([0.05, 0.05, 0.05])
    opt.point_size = 2.5
    
    vis.run()
    vis.destroy_window()


### VISUALIZAR MAPA DE SEMÁNTICO DE GT Y PREDICCIONES

In [3]:
import laspy
import numpy as np
import open3d as o3d
import torch

def visualizar_mapa_error(path_las, pred_pth):
    #Cargar el LAS y normalizar
    las = laspy.read(path_las)
    points = np.vstack((las.x, las.y, las.z)).T
    
    # Normalización para visualización óptima
    centro_xy = points[:, :2].mean(axis=0)
    z_min = points[:, 2].min()
    points[:, :2] -= centro_xy
    points[:, 2] -= z_min

    # Cargar Predicciones y generar máscara binaria
    preds = torch.load(pred_pth)
    num_puntos = points.shape[0]
    
    # Combinamos todas las instancias en una sola máscara de "Vegetación Predicha"
    is_tree_pred = np.zeros(num_puntos, dtype=bool)
    for i in range(preds['pred_masks'].shape[0]):
        mask = preds['pred_masks'][i].cpu().numpy().astype(bool)
        is_tree_pred |= mask # Operación OR para unir todos los árboles

    # Generar máscara binaria del Ground Truth (Clasificación >= 3)
    is_tree_gt = (las.classification >= 3)

    # Lógica de Colores (Mapa de Confusión)
    colors = np.full((num_puntos, 3), 0.15)  # Fondo gris oscuro por defecto

    # ACIERTO (True Positive): Verde
    # Ambos coinciden en que es árbol
    acierto_mask = is_tree_gt & is_tree_pred
    colors[acierto_mask] = [0.0, 1.0, 0.0]  # Verde brillante

    # ERROR (Falso Positivo o Falso Negativo): Rojo
    # El GT dice árbol y el Pred no, O el Pred dice árbol y el GT no
    error_mask = (is_tree_gt != is_tree_pred) & (is_tree_gt | is_tree_pred)
    colors[error_mask] = [1.0, 0.0, 0.0]    # Rojo puro

    # Crear Visualización en Open3D
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.colors = o3d.utility.Vector3dVector(colors)

    # Crear Visualización usando el Visualizer (permite fondo personalizado)
    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name="Análisis de Errores - Santomera", width=1200, height=800)
    
    # Añadimos la nube de puntos voxelizada
    vis.add_geometry(pcd)

    # CONFIGURACIÓN ESTÉTICA: Aquí es donde se define el fondo
    opt = vis.get_render_option()
    opt.background_color = np.asarray([0.05, 0.05, 0.05]) # Fondo gris oscuro
    opt.point_size = 2.5 # Ajusta el tamaño de los puntos si los ves muy pequeños
    
    print("Mapa de Errores Generado:")
    print(f"Verde: Acierto (Coinciden GT y Pred)")
    print(f"Rojo: Error (Inconsistencia entre GT y Pred)")

    vis.run()
    vis.destroy_window()

### VISUALIZAR REPRESENTACIÓN 3D DE UN ÁRBOL

In [4]:
import laspy
import numpy as np
import open3d as o3d
import torch

def visualizar_unico_arbol_alpha(path_las, pred_pth, indice_arbol=0, alpha_val=0.2):
    try:
        preds = torch.load(pred_pth)
    except Exception as e:
        print(f"Error al cargar el archivo .pth: {e}")
        return

    # 1. Cargar coordenadas del LAS
    las = laspy.read(path_las)
    points_all = np.vstack((las.x, las.y, las.z)).T

    # 2. Aislar los puntos del árbol seleccionado
    # Usamos la máscara de la predicción
    mask = preds['pred_masks'][indice_arbol].cpu().numpy().astype(bool)
    puntos_arbol = points_all[mask]

    if len(puntos_arbol) < 4:
        print("El árbol seleccionado no tiene suficientes puntos para crear una geometría 3D.")
        return

    # 3. CENTRADO CRÍTICO: Ponemos el árbol en el origen (0,0,0)
    # Esto evita que la cámara de Open3D aparezca lejos del objeto
    centro_arbol = puntos_arbol.mean(axis=0)
    puntos_arbol = puntos_arbol - centro_arbol

    # 4. Crear objeto de nube de puntos
    pcd_arbol = o3d.geometry.PointCloud()
    pcd_arbol.points = o3d.utility.Vector3dVector(puntos_arbol)
    pcd_arbol.paint_uniform_color([0.5, 0.5, 0.5]) # Puntos en gris suave

    # 5. Generar la geometría Alpha Shape 3D
    # Esta es la representación geométrica que pediste
    mesh_alpha = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd_arbol, alpha_val)
    mesh_alpha.compute_vertex_normals()
    
    # Estética de la malla
    mesh_alpha.paint_uniform_color([0.1, 0.8, 0.2]) # Color verde bosque

    # 6. Visualización
    print(f"Visualizando Árbol ID:{indice_arbol} con Alpha:{alpha_val}")
    print(f"Altura detectada: {puntos_arbol[:,2].max() - puntos_arbol[:,2].min():.2f} m")
    
    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name=f"Geometría Alpha Shape - Árbol {indice_arbol}", width=1000, height=800)
    
    # Añadimos la malla (geometría) y los puntos (opcional, para referencia)
    vis.add_geometry(mesh_alpha)
    vis.add_geometry(pcd_arbol)
    
    opt = vis.get_render_option()
    opt.background_color = np.asarray([0.05, 0.05, 0.05])
    opt.point_size = 3.0
    # Activamos el modo wireframe si quieres ver la estructura de triángulos
    opt.mesh_show_wireframe = True 
    
    vis.run()
    vis.destroy_window()


## Visualizar todos los árboles en representación 3D

In [5]:
import laspy
import numpy as np
import open3d as o3d
import torch

def visualizar_bosque_alpha(path_las, pred_pth, alpha_val=0.15):
    try:
        preds = torch.load(pred_pth)
    except Exception as e:
        print(f"Error al cargar el archivo .pth: {e}")
        return

    # 1. Cargar coordenadas y colores del LAS
    las = laspy.read(path_las)
    points_all = np.vstack((las.x, las.y, las.z)).T
    
    # Extraer RGB (Red, Green, Blue)
    try:
        # Algunos archivos LAS usan 16-bit (0-65535), normalizamos a 0-1
        red = np.array(las.red)
        green = np.array(las.green)
        blue = np.array(las.blue)
        
        # Detectar si es 16-bit o 8-bit
        max_val = 65535 if np.max(red) > 255 else 255
        rgb_all = np.vstack((red, green, blue)).T / max_val
    except AttributeError:
        print("⚠️ El archivo LAS no contiene información de color (RGB).")
        return

    # Centrado de la escena para Open3D
    centro_escena = points_all.mean(axis=0)
    points_all_centered = points_all - centro_escena

    num_instancias = preds['pred_masks'].shape[0]
    geometrias = []

    print(f"Calculando colores reales para {num_instancias} instancias...")

    # 2. Bucle por cada árbol
    for i in range(num_instancias):
        mask = preds['pred_masks'][i].cpu().numpy().astype(bool)
        
        puntos_arbol = points_all_centered[mask]
        colores_arbol = rgb_all[mask]

        if len(puntos_arbol) < 10:
            continue

        # --- CÁLCULO DEL COLOR PROMEDIO ---
        # Hacemos la media de R, G y B de todos los puntos de este árbol
        color_promedio = np.mean(colores_arbol, axis=0)

        # 3. Generar la malla Alpha Shape
        pcd_temp = o3d.geometry.PointCloud()
        pcd_temp.points = o3d.utility.Vector3dVector(puntos_arbol)
        
        try:
            mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(pcd_temp, alpha_val)
            mesh.compute_vertex_normals()
            
            # Aplicamos el color promedio real del árbol
            mesh.paint_uniform_color(color_promedio)
            geometrias.append(mesh)
        except:
            continue

    # 4. Visualización
    print(f"Mostrando {len(geometrias)} árboles con sus colores reales.")
    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name="Santomera: Gemelo Digital (Alpha Shapes RGB)", width=1200, height=800)
    
    for g in geometrias:
        vis.add_geometry(g)
    
    opt = vis.get_render_option()
    opt.background_color = np.asarray([0.05, 0.05, 0.05])
    opt.mesh_show_wireframe = False # Desactivamos para que se vea más realista
    opt.light_on = True # Activamos luces para dar volumen a las mallas
    
    vis.run()
    vis.destroy_window()

In [16]:
import os

# Ejecución
base_path_preds = "../../data/Santomera/result/OACNN_0"
base_path_gt = "../../data/Santomera/tiles/trees/labeled"

for file in os.listdir(base_path_preds):
    if file.endswith(".pth"):
        pred_path = os.path.join(base_path_preds, file)
        las_name = file.replace("_pred.pth", ".las")
        las_path = os.path.join(base_path_gt, las_name)
        print(f"Visualizando: {las_name} con predicción {file}")
        visualizacion_lado_a_lado(las_path, pred_path)

Visualizando: tile_2260.las con predicción tile_2260_pred.pth
Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)
Visualizando: tile_2303.las con predicción tile_2303_pred.pth
Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)
Visualizando: tile_2438.las con predicción tile_2438_pred.pth
Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)
Visualizando: tile_3330.las con predicción tile_3330_pred.pth
Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)
Visualizando: tile_3745.las con predicción tile_3745_pred.pth
Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)
Visualizando: tile_3747.las con predicción tile_3747_pred.pth
Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)
Visualizando: tile_3770.las con predicción tile_3770_pred.pth
Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)
Visualizando: tile_3771.las con predicción tile_3771_pred.pth
Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)
Visualizando: ti

In [17]:
import os

# Ejecución
base_path_preds = "../../data/Santomera/result/OACNN"
base_path_gt = "../../data/Santomera/tiles/trees/labeled"

for file in os.listdir(base_path_preds):
    if file.endswith(".pth"):
        pred_path = os.path.join(base_path_preds, file)
        las_name = file.replace("_pred.pth", ".las")
        las_path = os.path.join(base_path_gt, las_name)
        print(f"Visualizando: {las_name} con predicción {file}")
        visualizar_mapa_error(las_path, pred_path)

Visualizando: tile_2303.las con predicción tile_2303_pred.pth
Mapa de Errores Generado:
Verde: Acierto (Coinciden GT y Pred)
Rojo: Error (Inconsistencia entre GT y Pred)
Visualizando: tile_3745.las con predicción tile_3745_pred.pth
Mapa de Errores Generado:
Verde: Acierto (Coinciden GT y Pred)
Rojo: Error (Inconsistencia entre GT y Pred)
Visualizando: tile_3770.las con predicción tile_3770_pred.pth
Mapa de Errores Generado:
Verde: Acierto (Coinciden GT y Pred)
Rojo: Error (Inconsistencia entre GT y Pred)
Visualizando: tile_3771.las con predicción tile_3771_pred.pth
Mapa de Errores Generado:
Verde: Acierto (Coinciden GT y Pred)
Rojo: Error (Inconsistencia entre GT y Pred)
Visualizando: tile_3859.las con predicción tile_3859_pred.pth
Mapa de Errores Generado:
Verde: Acierto (Coinciden GT y Pred)
Rojo: Error (Inconsistencia entre GT y Pred)
Visualizando: tile_98.las con predicción tile_98_pred.pth
Mapa de Errores Generado:
Verde: Acierto (Coinciden GT y Pred)
Rojo: Error (Inconsistencia e

In [16]:
import os

# Ejecución
base_path_preds = "../../data/Santomera/result/OACNN"
base_path_gt = "../../data/Santomera/tiles/trees/labeled"

for file in os.listdir(base_path_preds):
    if file.endswith(".pth"):
        pred_path = os.path.join(base_path_preds, file)
        las_name = file.replace("_pred.pth", ".las")
        las_path = os.path.join(base_path_gt, las_name)
        print(f"Visualizando: {las_name} con predicción {file}")
        visualizar_unico_arbol_alpha(las_path, pred_path, indice_arbol=2, alpha_val=0.5)

Visualizando: tile_2303.las con predicción tile_2303_pred.pth
Visualizando Árbol ID:2 con Alpha:0.5
Altura detectada: 7.25 m
Visualizando: tile_3745.las con predicción tile_3745_pred.pth
Visualizando Árbol ID:2 con Alpha:0.5
Altura detectada: 8.35 m
Visualizando: tile_3770.las con predicción tile_3770_pred.pth
Visualizando Árbol ID:2 con Alpha:0.5
Altura detectada: 3.63 m
Visualizando: tile_3771.las con predicción tile_3771_pred.pth
Visualizando Árbol ID:2 con Alpha:0.5
Altura detectada: 6.31 m
Visualizando: tile_3859.las con predicción tile_3859_pred.pth
Visualizando Árbol ID:2 con Alpha:0.5
Altura detectada: 13.31 m
Visualizando: tile_98.las con predicción tile_98_pred.pth
Visualizando Árbol ID:2 con Alpha:0.5
Altura detectada: 5.33 m


In [ ]:
import os

# Ejecución
base_path_preds = "../../data/Santomera/result/OACNN"
base_path_gt = "../../data/Santomera/tiles/trees/labeled"

for file in os.listdir(base_path_preds):
    if file.endswith(".pth"):
        pred_path = os.path.join(base_path_preds, file)
        las_name = file.replace("_pred.pth", ".las")
        las_path = os.path.join(base_path_gt, las_name)
        print(f"Visualizando: {las_name} con predicción {file}")
        visualizar_bosque_alpha(las_path, pred_path, alpha_val=0.4)

Visualizando: tile_2260.las con predicción tile_2260_pred.pth
Calculando colores reales para 19 instancias...
Mostrando 19 árboles con sus colores reales.
Visualizando: tile_2303.las con predicción tile_2303_pred.pth
Calculando colores reales para 30 instancias...


KeyboardInterrupt: 

## Visualización conference

In [50]:
import os

# Ejecución
base_path_preds = "../../data/Santomera/result/OACNN_100"
base_path_gt = "../../data/Santomera/tiles/trees/labeled"
tile = "tile_3330"

pred_path = os.path.join(base_path_preds, tile + "_pred.pth")

las_path = os.path.join(base_path_gt, tile + ".las")
print(f"Visualizando: {tile} con predicción {tile}_pred.pth")
visualizacion_lado_a_lado(las_path, pred_path)

Visualizando: tile_3330 con predicción tile_3330_pred.pth
Mostrando Lado a Lado: GT (Izquierda) | Predicción (Derecha)
